### Setup

In [ ]:
#!pip install statsmodels
#!pip install tensorflow
#!pip install torch torchvision

In [ ]:
# Import libraries 
# data manipulation
import pandas as pd
import statsmodels.api as sm

# Machine Learning
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_curve, roc_auc_score, classification_report
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

# Neural Networks
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from transformers import BertTokenizer, BertForSequenceClassification
from transformers import RobertaTokenizer, RobertaForSequenceClassification

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Set a fixed seed value for reproducibility
seed = 6

# Target & text variable names
target_variable = 'onrust'
text_variable = 'rapportage'

### Data preparation

In [ ]:
# Load training and validation data
df_train = pd.read_csv('../zorgdata/df_train.csv', index_col=False)
df_valid = pd.read_csv('../zorgdata/df_valid.csv', index_col=False)

In [ ]:
# Selecti variables for the models. Excluded variables are not used in model training
excluded = ['ct_id', 'datum', 'rapportage', 'onrustscore', 'rapportage_clean', 
            'rapportage_len_chars', 'rapportage_clean_len_chars', 'rapportage_clean_len_words', 'discipline']

label = target_variable # Target variable

# Prepare training and validation sets by dropping excluded variables
X_train = df_train.drop(excluded + [label], axis=1)
y_train = df_train[label]

X_valid = df_valid.drop(excluded + [label], axis=1)
y_valid = df_valid[label]

# Add constant term for logistic regression
X_train_c = X_train.assign(const=1)

### Logistisc regression

In [ ]:
# Train the logistic regression model
lr_model = sm.Logit(y_train, X_train_c)
result = lr_model.fit()

# Calculate initial AIC for model comparison
initial_aic = result.aic

# Perform backward elimination based on p-values and AIC
while True:
    max_p_value = result.pvalues.max()

    if max_p_value > 0.05:
        variable_to_remove = result.pvalues.idxmax()
        X_train_c = X_train_c.drop(variable_to_remove, axis=1)
        
        # Gebruik sm.Logit consistent
        lr_model = sm.Logit(y_train, X_train_c)
        result = lr_model.fit()
        
        # Bereken de nieuwe AIC
        new_aic = result.aic
        
        # Stop als de AIC niet meer verbetert
        if new_aic >= initial_aic:
            break
        else:
            initial_aic = new_aic
    else:
        # Als alle p-waarden onder de drempelwaarde liggen, stop dan de eliminatie
        break

# Bekijk de uiteindelijke samenvatting van het model
print(result.summary2())

In [ ]:
# Prepare validation data with selected variables
selected_variables = X_train_c.columns.tolist()
X_valid_c = df_valid.assign(const=1)[selected_variables]

In [ ]:
# Make predictions on validation set and adding them to a new dataframe

# Create new df with predictions
df_valid_pred = df_valid[['ct_id', 'datum', 'onrustscore', target_variable, 'rapportage']].copy()

# Add predicted probability and predicted label
df_valid_pred['pred_lr_prob'] = result.predict(X_valid_c)
df_valid_pred['pred_lr'] = (df_valid_pred['pred_lr_prob'] >= 0.5).astype(int)

### Random forest

In [ ]:
# # Uncomment below to perform hyperparameter tuning using GridSearchCV
# param_grid = {
#     'n_estimators': [100, 501],
#     'max_depth': [None, 10, 30],
#     'min_samples_split': [2, 10],
#     'min_samples_leaf': [1, 4]
# }

# # Initialize Grid Search with cross-validation
# grid_search = GridSearchCV(estimator=RandomForestClassifier(random_state=seed), 
#                            param_grid=param_grid, 
#                            cv=5, # Aantal folds voor crossvalidaion
#                            n_jobs=-1, # Gebruik alle beschikbare CPU-cores
#                            verbose=2, # Toon gedetailleerde voortgangsinformatie
#                            scoring='roc_auc') # AUC als prestatie-indicator

# # Perform Grid Search with training data
# grid_search.fit(X_train, y_train)

# # Best parameters and AUC-score
# print("Beste hyperparameters:", grid_search.best_params_)
# print("Beste AUC-score:", grid_search.best_score_)

# # Best model
# best_rf_model = grid_search.best_estimator_

In [ ]:
# Train the Random Forest model with specified parameters
rf_model = RandomForestClassifier(n_estimators=501, 
                                  random_state=seed, 
                                  max_depth=None, 
                                  min_samples_leaf=4, 
                                  min_samples_split=2)
rf_model.fit(X_train, y_train)

In [ ]:
# Make predictions on validation set
df_valid_pred['pred_rf'] = rf_model.predict(X_valid)
df_valid_pred['pred_rf_prob'] = rf_model.predict_proba(X_valid)[:, 1]

In [ ]:
# Display feature importances
feature_importances = pd.DataFrame(rf_model.feature_importances_,
                                   index = X_train.columns,
                                   columns=['importance']).sort_values('importance', ascending=False)
print(feature_importances[0:10])

### TensorFlow Neural Network

In [ ]:
# Normalize the features for TensorFlow model
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

In [ ]:
# Construct the TensorFlow neural network
tf_model = Sequential()
tf_model.add(Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)))
tf_model.add(Dense(32, activation='relu'))
tf_model.add(Dense(1, activation='sigmoid')) # Uitvoerlaag voor binaire classificatie

In [ ]:
# Compile the TensorFlow model
tf_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
# Train the TensorFlow model
history = tf_model.fit(X_train_scaled, y_train, epochs=10, batch_size=32, validation_data=(X_valid_scaled, y_valid))

In [ ]:
# Evaluate the TensorFlow model
loss, accuracy = tf_model.evaluate(X_valid_scaled, y_valid)
print(f'Validatie nauwkeurigheid: {accuracy:.2f}, Verlies: {loss:.2f}')

In [ ]:
# Predict probabilities for the validation set using TensorFlow model
df_valid_pred['pred_tf_prob'] = tf_model.predict(X_valid_scaled)
df_valid_pred['pred_tf'] = (df_valid_pred['pred_tf_prob'] > 0.5).astype("int32")

### PyTorch Neural Network

In [ ]:
# Prepare the data for PyTorch model
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

# Convert data to PyTorch tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32)
X_valid_tensor = torch.tensor(X_valid_scaled, dtype=torch.float32)
y_valid_tensor = torch.tensor(y_valid.values, dtype=torch.float32)

# Create datasets and DataLoaders for PyTorch
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
valid_dataset = TensorDataset(X_valid_tensor, y_valid_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False)

In [ ]:
# Define the PyTorch neural network
class NeuralNet(nn.Module):
    def __init__(self):
        super(NeuralNet, self).__init__()
        self.fc1 = nn.Linear(X_train_scaled.shape[1], 64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.sigmoid(self.fc3(x))
        return x

# Initialize model, loss function, and optimizer for PyTorch
pt_model = NeuralNet()
criterion = nn.BCELoss()
optimizer = optim.Adam(pt_model.parameters(), lr=0.001)

In [ ]:
# Train the PyTorch model
num_epochs = 10
for epoch in range(num_epochs):
    pt_model.train()
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        y_pred = pt_model(X_batch)
        loss = criterion(y_pred.squeeze(), y_batch)
        loss.backward()
        optimizer.step()

    # Validate the PyTorch model
    pt_model.eval()
    with torch.no_grad():
        correct = 0
        total = 0
        for X_batch, y_batch in valid_loader:
            y_pred = pt_model(X_batch)
            predicted = (y_pred.squeeze() > 0.5).float()
            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()

    accuracy = 100 * correct / total
    print(f'Epoch {epoch+1}, Verlies: {loss.item():.4f}, Nauwkeurigheid: {accuracy:.2f}%')

In [ ]:
# Calculate predicted probabilities and labels using PyTorch model
pt_model.eval()  # Set the model to evaluation mode
with torch.no_grad():
    df_valid_pred['pred_pt_prob'] = pt_model(X_valid_tensor).numpy().ravel()

# Convert the predicted probabilities to labels
df_valid_pred['pred_pt'] = (df_valid_pred['pred_pt_prob'] >= 0.5)

### Bert

In [ ]:
# Path where the model is saved
model_path = '../models/bert_classification' 

# Load the finetuned Bert model
bc_model = BertForSequenceClassification.from_pretrained(model_path)
bc_tokenizer = BertTokenizer.from_pretrained('wietsedv/bert-base-dutch-cased')

In [ ]:
# Get validation encodings for the Bert model
validation_encodings = bc_tokenizer(df_valid_pred[text_variable].tolist(), truncation=True, padding=True, max_length=512, return_tensors="pt")

In [ ]:
# Predict probabilities with the Bert model
bc_model.eval()  
with torch.no_grad():
    logits = bc_model(**validation_encodings).logits
    probabilities = torch.nn.functional.softmax(logits, dim=1)
    df_valid_pred['pred_bert_prob'] = probabilities[:, 1].numpy()  # Probability of class '1'

# Convert the probabilities to labels
df_valid_pred['pred_bert'] = (df_valid_pred['pred_bert_prob'] >= 0.5)

### Roberta

In [ ]:
# Path where the model is saved
model_path = '../models/RobertaClassification' 

# Load the finetuned RoBerta model
rb_model = RobertaForSequenceClassification.from_pretrained(model_path)
rb_tokenizer = RobertaTokenizer.from_pretrained('pdelobelle/robbert-v2-dutch-base')

In [ ]:
# Get validation encodings for the Roberta model
validation_encodings = rb_tokenizer(df_valid_pred[text_variable].tolist(), truncation=True, padding=True, max_length=512, return_tensors="pt")

In [ ]:
# Predict probabilities with the Roberta model
rb_model.eval()  
with torch.no_grad():
    logits = rb_model(**validation_encodings).logits
    probabilities = torch.nn.functional.softmax(logits, dim=1)
    df_valid_pred['pred_roberta_prob'] = probabilities[:, 1].numpy()  # Probability of class '1'

# Convert the probabilities to labels
df_valid_pred['pred_roberta'] = (df_valid_pred['pred_roberta_prob'] >= 0.5)

### MedRoberta

In [ ]:
# Path where the model is saved
model_path = '../models/MedRobertaClassification' 

# Load the finetuned MedRoBerta model
mrb_model = RobertaForSequenceClassification.from_pretrained(model_path)
mrb_tokenizer = RobertaTokenizer.from_pretrained('CLTL/MedRoBERTa.nl')

In [ ]:
# Get validation encodings for the Roberta model
validation_encodings = mrb_tokenizer(df_valid_pred[text_variable].tolist(), truncation=True, padding=True, max_length=512, return_tensors="pt")

In [ ]:
# Predict probabilities with the MedRoberta model
mrb_model.eval()  
with torch.no_grad():
    logits = mrb_model(**validation_encodings).logits
    probabilities = torch.nn.functional.softmax(logits, dim=1)
    df_valid_pred['pred_medroberta_prob'] = probabilities[:, 1].numpy()  # Probability of class '1'

# Convert the probabilities to labels
df_valid_pred['pred_medroberta'] = (df_valid_pred['pred_medroberta_prob'] >= 0.5)

### Evaluate

In [ ]:
df_valid_pred.head()

In [ ]:
def plot_contingency(data, model_name, subplot):
    # Create a cross-tabulation heatmap for given model predictions vs actual values
    kruistabel = pd.crosstab(data[target_variable], data[model_name], 
                             rownames=['Werkelijke Onrust'], colnames=['Voorspelde Onrust'])
    sns.heatmap(kruistabel, annot=True, fmt='d', cmap='Blues', ax=subplot)
    subplot.set_title(model_name)

# Prepare figure and define subplots for visual comparison
fig, axs = plt.subplots(1, 7, figsize=(20, 4))  # 5 compare side-by-side

# Plot each model's cross-tabulation in its own subplot
plot_contingency(df_valid_pred, 'pred_lr', axs[0])
plot_contingency(df_valid_pred, 'pred_rf', axs[1])
plot_contingency(df_valid_pred, 'pred_tf', axs[2])
plot_contingency(df_valid_pred, 'pred_pt', axs[3])
plot_contingency(df_valid_pred, 'pred_bert', axs[4])
plot_contingency(df_valid_pred, 'pred_roberta', axs[5])
plot_contingency(df_valid_pred, 'pred_medroberta', axs[6])

plt.tight_layout()
plt.show()

In [ ]:
# Calculate ROC curves for each model
fpr_lr, tpr_lr, thresholds_lr = roc_curve(df_valid_pred[target_variable], df_valid_pred['pred_lr_prob'])
fpr_rf, tpr_rf, thresholds_rf = roc_curve(df_valid_pred[target_variable], df_valid_pred['pred_rf_prob'])
fpr_tf, tpr_tf, thresholds_tf = roc_curve(df_valid_pred[target_variable], df_valid_pred['pred_tf_prob'])
fpr_pt, tpr_pt, thresholds_pt = roc_curve(df_valid_pred[target_variable], df_valid_pred['pred_pt_prob'])
fpr_bert, tpr_bert, thresholds_bert = roc_curve(df_valid_pred[target_variable], df_valid_pred['pred_bert_prob'])
fpr_roberta, tpr_roberta, thresholds_roberta = roc_curve(df_valid_pred[target_variable], df_valid_pred['pred_roberta_prob'])
fpr_medroberta, tpr_medroberta, thresholds_medroberta = roc_curve(df_valid_pred[target_variable], df_valid_pred['pred_medroberta_prob'])

# Calculate Area Under Curve (AUC) for each model
roc_auc_lr = roc_auc_score(df_valid_pred[target_variable], df_valid_pred['pred_lr_prob'])
roc_auc_rf = roc_auc_score(df_valid_pred[target_variable], df_valid_pred['pred_rf_prob'])
roc_auc_tf = roc_auc_score(df_valid_pred[target_variable], df_valid_pred['pred_tf_prob'])
roc_auc_pt = roc_auc_score(df_valid_pred[target_variable], df_valid_pred['pred_pt_prob'])
roc_auc_bert = roc_auc_score(df_valid_pred[target_variable], df_valid_pred['pred_bert_prob'])
roc_auc_roberta = roc_auc_score(df_valid_pred[target_variable], df_valid_pred['pred_roberta_prob'])
roc_auc_medroberta = roc_auc_score(df_valid_pred[target_variable], df_valid_pred['pred_medroberta_prob'])

plt.figure(figsize=(10, 4))

# Plot the ROC Curves for each model
plt.plot(fpr_lr, tpr_lr, label='Logistisch Regressie (AUC = {:.2f})'.format(roc_auc_lr), color='blue', linestyle='-')
plt.plot(fpr_rf, tpr_rf, label='Random Forest (AUC = {:.2f})'.format(roc_auc_rf), color='green', linestyle='--')
plt.plot(fpr_tf, tpr_tf, label='Neuraal Netwerk (TensorFlow) (AUC = {:.2f})'.format(roc_auc_tf), color='red', linestyle='-.')
plt.plot(fpr_pt, tpr_pt, label='Neuraal Netwerk (PyTorch) (AUC = {:.2f})'.format(roc_auc_pt), color='purple', linestyle=':')
plt.plot(fpr_bert, tpr_bert, label='Bert (AUC = {:.2f})'.format(roc_auc_bert), color='orange', linestyle='-')
plt.plot(fpr_roberta, tpr_roberta, label='Roberta (AUC = {:.2f})'.format(roc_auc_roberta), color='grey', linestyle='--')
plt.plot(fpr_medroberta, tpr_medroberta, label='MedRoberta (AUC = {:.2f})'.format(roc_auc_medroberta), color='magenta', linestyle='--')

# Baseline for comparison
plt.plot([0, 1], [0, 1], 'k--')

# Axes and title
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve Vergelijking', fontsize=14)

# Legend
plt.legend(loc='lower right', fontsize=10)

# Grid for readability
plt.grid(True)

# Set axis limits
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.0])

plt.show()

In [ ]:
# Predict on custom text
custom_text = "Hij heeft me geslagen"

# Tokenize the text for Bert model
inputs = bc_tokenizer(custom_text, padding=True, truncation=True, max_length=512, return_tensors="pt")

# Predict using the Bert model
bc_model.eval()
with torch.no_grad():
    outputs = bc_model(**inputs)
    logits = outputs.logits

# Convert logits to probabilities
probabilities = torch.nn.functional.softmax(logits, dim=1)

# Extract the highest probability and its corresponding label
predicted_prob, predicted_label = torch.max(probabilities, dim=1)
predicted_prob = predicted_prob.item() # Convert to a Python number
predicted_label = predicted_label.item() # Convert to a Python number

print(f"Predicted Label: {predicted_label}, Predicted Probability: {predicted_prob:.2f}")


In [ ]:
def evaluate_model_performance(actuals, predictions, model_name="Model"):
    """
    Evaluates the performance of a binary classification model including specificity and NPV.

    Parameters:
    actuals (array-like): The actual labels.
    predictions (array-like): The predicted labels.
    model_name (str): Name of the model for display purposes.

    Returns:
    None: Prints the performance metrics.
    """
    # Calculate the confusion matrix
    tn, fp, fn, tp = confusion_matrix(actuals, predictions).ravel()

    # Calculate the metrics
    accuracy = accuracy_score(actuals, predictions)
    precision = precision_score(actuals, predictions)  # Positive Predictive Value (PPV)
    recall = recall_score(actuals, predictions)  # Sensitivity or True Positive Rate
    f1 = f1_score(actuals, predictions)
    specificity = tn / (tn + fp)
    npv = tn / (tn + fn)  # Negative Predictive Value (NPV)

    model_name = model_name.ljust(20)

    # Print the results
    print(f"{model_name} - Accuracy: {accuracy:.3f}, Precision (PPV): {precision:.2f}, NPV: {npv:.2f}, Recall (Sensitivity): {recall:.2f}, Specificity: {specificity:.2f}, F1 Score: {f1:.3f}")


In [ ]:
evaluate_model_performance(df_valid_pred[target_variable], df_valid_pred['pred_lr'], 'Logistic regression')
evaluate_model_performance(df_valid_pred[target_variable], df_valid_pred['pred_rf'], 'Random Forest')
evaluate_model_performance(df_valid_pred[target_variable], df_valid_pred['pred_tf'], 'TensorFlow')
evaluate_model_performance(df_valid_pred[target_variable], df_valid_pred['pred_pt'], 'PyTorch')
evaluate_model_performance(df_valid_pred[target_variable], df_valid_pred['pred_bert'], 'Bert llm')
evaluate_model_performance(df_valid_pred[target_variable], df_valid_pred['pred_roberta'], 'Roberta llm')
evaluate_model_performance(df_valid_pred[target_variable], df_valid_pred['pred_medroberta'], 'MedRoberta llm')

In [ ]:
#df_valid_pred.to_csv('../zorgdata/df_valid_pred.csv')

In [ ]:
# Predict on custom text
custom_text = "Meneer liep vandaag veel heen en weer"

# Tokenize the text for MedRoberta model
inputs = mrb_tokenizer(custom_text, padding=True, truncation=True, max_length=512, return_tensors="pt")

# Predict using the Bert model
mrb_model.eval()
with torch.no_grad():
    outputs = mrb_model(**inputs)
    logits = outputs.logits

# Convert logits to probabilities
probabilities = torch.nn.functional.softmax(logits, dim=1)

# Extract the highest probability and its corresponding label
predicted_prob, predicted_label = torch.max(probabilities, dim=1)
predicted_prob = predicted_prob.item() # Convert to a Python number
predicted_label = predicted_label.item() # Convert to a Python number

print(f"Predicted Label: {predicted_label}, Predicted Probability: {predicted_prob:.2f}")
